# Importation librairie 

In [52]:
import tensorflow as tf
#import tensorflow_decision_forests as tfdf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC 
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
#import xgboost as xgb
#from catboost import CatBoostClassifier
#from lightgbm import LGBMClassifier

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler


train_df = pd.read_csv("train.csv")
train_df_copy = train_df.copy()
test_df = pd.read_csv("test.csv")
test_df_copy = test_df.copy()
print("Full train dataset shape is {}".format(train_df.shape))

Full train dataset shape is (8693, 14)


# Toutes les modifications de nos données

## Ajout de nouvelles variables

In [53]:
def age_group(df):
    age_group  = []
    for i in df["Age"]:
        if i<=4:
            age_group.append("Age_0-4")
        elif (i>4 and i<=12):
            age_group.append("Age_05-12")
        elif (i>12 and i<=18):
            age_group.append("Age_13-18")
        elif (i>18 and i<=25):
            age_group.append("Age_19-25")
        elif (i>25 and i<=32):
            age_group.append("Age_26-32")
        elif (i>32 and i<=50):
            age_group.append("Age_33_50")
        elif (i>50):
            age_group.append("Age_50+")
        else:
            age_group.append(np.nan)
        
    df["Age Group"] = age_group

age_group(train_df)
age_group(test_df)


def passagerid_new_features(df):
    df["Group"] = df["PassengerId"].apply(lambda x: int(x.split("_")[0]))
    df["Member"] = df["PassengerId"].apply(lambda x: int(x.split("_")[1]))

    x = df.groupby("Group")["Member"].count()
    y = set(x[x>1].index)

    df["Travelling_Solo"] = df["Group"].apply(lambda x : x not in y)
    df["Group_size"] = 0

    for i in x.items():
        df.loc[df["Group"]==i[0], "Group_size"] = i[1]
    df["Group_size"] = df["Group_size"].astype(int)

passagerid_new_features(train_df)
passagerid_new_features(test_df)




def cabin_new_feature(df):
    df["Cabin"].fillna("np.nan/np.nan/np.nan", inplace=True)
    
    df["Cabin_Deck"] = df["Cabin"].apply(lambda x: x.split("/")[0])
    df["Cabin_Number"] = df["Cabin"].apply(lambda x: x.split("/")[1])
    df["Cabin_Side"] = df["Cabin"].apply(lambda x: x.split("/")[2])
    
    # Remplacer les valeurs de chaîne 'np.nan' par des valeurs NaN de numpy
    cols = ["Cabin_Deck", "Cabin_Number", "Cabin_Side"]
    df[cols] = df[cols].replace("np.nan", np.nan)
    
    # Remplir les valeurs manquantes dans les nouvelles caractéristiques créées
    df["Cabin_Deck"].fillna(df["Cabin_Deck"].mode()[0], inplace=True)
    df["Cabin_Side"].fillna(df["Cabin_Side"].mode()[0], inplace=True)
    df["Cabin_Number"] = pd.to_numeric(df["Cabin_Number"], errors='coerce')  # Conversion en numérique
    df["Cabin_Number"].fillna(df["Cabin_Number"].median(), inplace=True)

cabin_new_feature(train_df)
cabin_new_feature(test_df)





def cabin_regions(df):
    df["Cabin_Region1"] = (df["Cabin_Number"]<300)
    df["Cabin_Region2"] = (df["Cabin_Number"]>=300) & (df["Cabin_Number"]<600)
    df["Cabin_Region3"] = (df["Cabin_Number"]>=600) & (df["Cabin_Number"]<900)
    df["Cabin_Region4"] = (df["Cabin_Number"]>=900) & (df["Cabin_Number"]<1200)
    df["Cabin_Region5"] = (df["Cabin_Number"]>=1200) & (df["Cabin_Number"]<1500)
    df["Cabin_Region6"] = (df["Cabin_Number"]>=1500)

cabin_regions(train_df)
cabin_regions(test_df)





exp_cols = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]

def new_exp_features(df):
    df["Total Expenditure"] = df[exp_cols].sum(axis=1)
    df["No Spending"] = (df["Total Expenditure"]==0)

new_exp_features(train_df)
new_exp_features(test_df)

meanExp = round(train_df["Total Expenditure"].mean())
medianExp = train_df["Total Expenditure"].median()

def expenditure_category(df):
    expense_category = []   
    for i in df["Total Expenditure"]:
        if i==0:
            expense_category.append("No Expense")
        elif (i>0 and i<=meanExp):
            expense_category.append("Low Expense")
        elif (i>meanExp and i<=medianExp):
            expense_category.append("Medium Expense")
        elif (i>medianExp):
            expense_category.append("High Expense")
    df["Expenditure Category"] = expense_category

expenditure_category(train_df)
expenditure_category(test_df)

def is_in_a_group(row):
    if pd.isnull(row['Group']):
        return 0
    else:
        return 1
train_df['InGroup'] = train_df.apply(is_in_a_group, axis=1)
test_df['InGroup'] = test_df.apply(is_in_a_group, axis=1)

def fill_destination(df):
    # Remplir les valeurs manquantes dans chaque groupe avec la destination la plus fréquente dans ce groupe
    df['Destination'] = df.groupby('Group')['Destination'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

    # Remplir les valeurs manquantes restantes avec la destination la plus fréquente dans l'ensemble du DataFrame
    df['Destination'] = df['Destination'].fillna(df['Destination'].mode()[0])
fill_destination(train_df)
fill_destination(test_df)

In [54]:
train_df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,Cabin_Region1,Cabin_Region2,Cabin_Region3,Cabin_Region4,Cabin_Region5,Cabin_Region6,Total Expenditure,No Spending,Expenditure Category,InGroup
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,True,False,False,False,False,False,0.0,True,No Expense,1
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,...,True,False,False,False,False,False,736.0,False,Medium Expense,1
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,...,True,False,False,False,False,False,10383.0,False,High Expense,1
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,...,True,False,False,False,False,False,5176.0,False,High Expense,1
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,...,True,False,False,False,False,False,1091.0,False,Medium Expense,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,...,True,False,False,False,False,False,8536.0,False,High Expense,1
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,...,False,False,False,False,True,False,0.0,True,No Expense,1
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,...,False,False,False,False,False,True,1873.0,False,High Expense,1
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,...,False,False,True,False,False,False,4637.0,False,High Expense,1


## Remplissage des données manquantes

Cette partie est encore très naïve. IL faudra sûrement s'y attarder davantage dans le futur pour améliorer la précision de notre modèle.

In [37]:
#cat_cols = train_df.select_dtypes(include=["object","bool"]).columns.tolist()
#cat_cols.remove("Transported")
#num_cols = train_df.select_dtypes(include=["int","float"]).columns.tolist()
#
#def fill_missingno(df):
#    df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])
#    df[num_cols] = SimpleImputer(strategy="median").fit_transform(df[num_cols])
#
#fill_missingno(train_df)
#fill_missingno(test_df)
#
def complete_missing_values_advanced(df):
    columns_with_missing_values = [col for col in df.columns if df[col].isnull().any()]
    
    for col in columns_with_missing_values:
        # Vérifier si la colonne est numérique ou catégorielle
        if df[col].dtype == 'object':
            # Pour les colonnes catégorielles, utiliser le mode du groupe
            fill_value = df.groupby('Group')[col].transform(lambda x: x.mode()[0] if not x.mode().empty else x)
        else:
            # Pour les colonnes numériques, utiliser la médiane du groupe
            fill_value = df.groupby('Group')[col].transform(lambda x: x.fillna(x.median()))
        
        # Remplir les valeurs manquantes
        df[col] = df[col].fillna(fill_value)
        
        # Pour les valeurs qui restent NA (si le groupe entier était NA), remplir avec la médiane/mode globale
        if df[col].isnull().any():
            if df[col].dtype == 'object':
                df[col] = df[col].fillna(df[col].mode()[0])
            else:
                df[col] = df[col].fillna(df[col].median())
        
    return df

# Appliquer la fonction améliorée aux DataFrames
train_df = complete_missing_values_advanced(train_df)
test_df = complete_missing_values_advanced(test_df)


## Suppression des variables maintenant inutiles

In [56]:
pass_df = test_df[["PassengerId"]]
cols = ["PassengerId","Group","Member", "Cabin","Name","Cabin_Number","InGroup"]

train_df.drop(columns=["Group","Member"],inplace=True)
test_df.drop(columns=["Group","Member"],inplace=True)

train_df.drop(columns =cols, inplace=True)
test_df.drop(columns=cols, inplace=True)


KeyError: "None of [Index(['PassengerId'], dtype='object')] are in the [columns]"

## Traitement final : Encodage One-Hot et Label Encoding

In [39]:
nominal_cat_cols_one_Hot = ["HomePlanet","Destination"]

train_df = pd.get_dummies(train_df, columns= nominal_cat_cols_one_Hot)
test_df = pd.get_dummies(test_df, columns = nominal_cat_cols_one_Hot)

ordinal_cat_cols_Label = ["CryoSleep","VIP","Travelling_Solo","Cabin_Deck","Cabin_Side","Cabin_Region1","Cabin_Region2",
                    "Cabin_Region3","Cabin_Region4","Cabin_Region5","Cabin_Region6","Age Group","No Spending",
                    "Expenditure Category"]

binary_cols = [col for col in train_df.columns if train_df[col].dropna().unique().size == 2]
new_binary_cols = [col for col in binary_cols if col not in ordinal_cat_cols_Label]

ordinal_cat_cols_Label.extend(new_binary_cols)
ordinal_cat_cols_Label.remove("Transported")

train_df[ordinal_cat_cols_Label] = train_df[ordinal_cat_cols_Label].apply(LabelEncoder().fit_transform)
test_df[ordinal_cat_cols_Label] = test_df[ordinal_cat_cols_Label].apply(LabelEncoder().fit_transform)

In [40]:
train_df.columns
train_df.to_csv('train_travaille.csv', index=False)

## Pour commencer à travailler

Il est temps d'initialiser X et Y, et on fait à présent un partage des données entre X_train et X_test.
On fait également une normalisation des données : on aura donc le choix entre X_train et X_train_scaled pour construire notre modèle de prédiction.

In [41]:
Y = train_df["Transported"]
X = train_df.drop(columns=["Transported"])

X_scaled = StandardScaler().fit_transform(X)
test_df_scaled = StandardScaler().fit_transform(test_df)

X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=0)

X_train_scaled, X_test_scaled, Y_train_scaled, Y_test_scaled = train_test_split(X_scaled,Y,test_size=0.2,random_state=0)
X

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Age Group,Group,...,Cabin_Region6,Total Expenditure,No Spending,Expenditure Category,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,0,39.0,0,0.0,0.0,0.0,0.0,0.0,5,1,...,0,0.0,1,3,0,1,0,0,0,1
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0,3,2,...,0,736.0,0,2,1,0,0,0,0,1
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,6,3,...,0,10383.0,0,0,0,1,0,0,0,1
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,5,3,...,0,5176.0,0,0,0,1,0,0,0,1
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0,2,4,...,0,1091.0,0,2,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,0,41.0,1,0.0,6819.0,0.0,1643.0,74.0,5,9276,...,0,8536.0,0,0,0,1,0,1,0,0
8689,1,18.0,0,0.0,0.0,0.0,0.0,0.0,2,9278,...,0,0.0,1,3,1,0,0,0,1,0
8690,0,26.0,0,0.0,0.0,1872.0,1.0,0.0,4,9279,...,1,1873.0,0,0,1,0,0,0,0,1
8691,0,32.0,0,0.0,1049.0,0.0,353.0,3235.0,4,9280,...,0,4637.0,0,0,0,1,0,1,0,0


# Les modèles

C'est là qu'on peut enfin tester nos modèles

In [42]:
from sklearn.linear_model import LogisticRegression

# Create an instance of Logistic Regression
regression_model = LogisticRegression(random_state=0)

# Fit the regression model using X_train_scaled and Y_train
regression_model.fit(X_train_scaled, Y_train)

# Predict the values of X_test_scaled
Y_pred = regression_model.predict(X_test_scaled)

# Print the accuracy of the model
print("Accuracy: ", accuracy_score(Y_test, Y_pred))

Accuracy:  0.7872340425531915


In [43]:
Y_test_pred = regression_model.predict(test_df_scaled)

#Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId':test_df_copy['PassengerId'],
    'Transported':Y_test_pred
})

resultat_df.to_csv('resultat.csv', index=False)

ça prend environ 1min8.5 pour tourner
(le bout en dessous)

In [44]:
from sklearn.model_selection import RandomizedSearchCV

model = RandomForestClassifier()

param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, None],
    'max_features': ['auto', 'sqrt'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

random_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1)

random_search.fit(X_train_scaled, Y_train)


Fitting 3 folds for each of 100 candidates, totalling 300 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:425: FitFailedWarning: 
156 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
96 fits failed with the following error:
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 732, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py", line 1144, in wrapper
    estimator._validate_params()
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py", line 637, in _validate_params
    validate_parameter_constraints(
  File "c:\ProgramData\anaconda3\Lib\site-package

RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(), n_iter=100,
                   n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [10, 20, 30, 40, 50, 60,
                                                      70, 80, 90, 100, None],
                                        'max_features': ['auto', 'sqrt'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 400,
                                                         500]},
                   random_state=42, verbose=2)

In [45]:
Y_test_pred_2 = random_search.predict(test_df_scaled)

#Créer le data frame de résultats
resultat_df2 = pd.DataFrame({
    'PassengerId':test_df_copy['PassengerId'],
    'Transported':Y_test_pred_2
})

resultat_df2.to_csv('resultat_2.csv', index=False)

<h2>Modèle SVM</h2>

<p>(j'ai arreter le prgromme car il prenait trop de temps à tourner, plus de 5 heures)</p>
-> réduction du nombre de valeurs pour kernel, C et gamma

Pb: avec kernel = poly on obtient des floats et donc la précision est de 0.0

In [46]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd

param_grid_svm ={
    'kernel':['linear','rbf', 'poly'],
    'C': [0.1, 0.5,0.7,0.9,1,2,3],
    'gamma': ['scale', 'auto']
}

# Initialiser le modèle
model_svm = SVC()

grid_search_svm = GridSearchCV(model_svm , param_grid_svm, cv=5, scoring='accuracy') #calcul le nb de bonne comparaison, le mieux pour une classification
# Entraîner le modèle
grid_search_svm.fit(X_train_scaled, Y_train) #scaled car calcule de distance

Y_test_pred = grid_search_svm.predict(test_df_scaled)

#Créer le data frame de résultats
resultat_df3 = pd.DataFrame({
    'PassengerId':test_df_copy['PassengerId'],
    'Transported':Y_test_pred
})

resultat_df3.to_csv('resultat_svc.csv', index=False)

In [51]:
# Supposons que grid_search_svm est l'objet GridSearchCV après l'entraînement
best_params = grid_search_svm.best_params_
best_score = grid_search_svm.best_score_
cv_results = grid_search_svm.cv_results_

# Pour afficher tous les résultats
all_results_df = pd.DataFrame(cv_results)

# Pour afficher les meilleurs résultats uniquement
best_results_df_svc = pd.DataFrame([{
    'SVC Best Score': best_score,
    'Kernel': best_params['kernel'],
    'C': best_params['C'],
    'Gamma': best_params['gamma']
}])
best_results_df_svc

,SVC Best Score,Kernel,C,Gamma
0,0.793214,linear,3,scale


<h3>Nearest Neighbors</h3>

precision = 0.74982 (avant opti des hyperparamètres)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

# Initialiser le modèle avec le nombre de voisins que vous voulez - par exemple, 3
model_knn = KNeighborsClassifier(n_neighbors=3)

# Entraîner le modèle
model_knn.fit(X_train_scaled, Y_train)

# Faire une prédiction
Y_test_pred = model_knn.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_knn.csv', index=False)

(opti des hyperparamètres) precision = 0.77367

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Initialiser le modèle
model_knn = KNeighborsClassifier()

# Définir les paramètres à optimiser
param_grid = {
    'n_neighbors': list(range(1, 15)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_knn, param_grid, cv=5)

# Entraîner le modèle
grid_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", grid_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_knn.csv', index=False)

KeyboardInterrupt: 

<h2>Logistic Regression</h2>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Initialiser le modèle
model_lr = LogisticRegression()

# Définir les paramètres à optimiser
param_grid = {
    'C': [0.01,1, 10, 100],
    'penalty': ['l1', 'l2'],
    'max_iter': list(range(100,800,100)),
    'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_lr, param_grid, cv=5)

# Entraîner le modèle
grid_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", grid_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_lr.csv', index=False)

c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the 

Meilleurs paramètres :  {'C': 1, 'max_iter': 200, 'penalty': 'l1', 'solver': 'saga'}


c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


<h2>Ensemble learning</h2>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Initialiser le modèle
model_rf = RandomForestClassifier()

# Définir les paramètres à optimiser
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth' : [4,5,6,7,8],
    'criterion' :['gini', 'entropy']
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_rf, param_grid, cv=5)

# Entraîner le modèle
print("Meilleurs paramètres : ", grid_search.best_params_)


# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_rf.csv', index=False)

c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
150 fits failed out of a total of 450.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
150 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "c:\Users\theot\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base

Meilleurs paramètres :  {'criterion': 'gini', 'max_depth': 8, 'max_features': 'sqrt', 'n_estimators': 200}


Optimisations des hyperparamètres avec une random search et en utilisant les info sur les meilleurs paramètres obtenue lors de la grid search

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd

# Initialiser le modèle
model_rf = RandomForestClassifier()

# Définir les paramètres à optimiser
param_grid = {
    'n_estimators': [150, 175, 200,225,250],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth' : list(range(1, 11)),
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False],
    'criterion' :['gini', 'entropy']
}

# Initialiser la recherche aléatoire
random_search = RandomizedSearchCV(model_rf, param_grid, cv=5, n_iter=100)

# Entraîner le modèle
random_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", random_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = random_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_rf.csv', index=False)

<h2>Bayesian</h2>

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd

# Initialiser le modèle
model_nb = GaussianNB()

# Définir les paramètres à optimiser
param_grid = {
    'var_smoothing': np.logspace(0,-9, num=100)
}

# Initialiser la recherche sur grille
grid_search = GridSearchCV(model_nb, param_grid, cv=5)

# Entraîner le modèle
grid_search.fit(X_train_scaled, Y_train)

# Afficher les meilleurs paramètres
print("Meilleurs paramètres : ", grid_search.best_params_)

# Faire une prédiction avec le meilleur modèle
Y_test_pred = grid_search.predict(test_df_scaled)

# Créer le data frame de résultats
resultat_df = pd.DataFrame({
    'PassengerId': test_df_copy['PassengerId'],
    'Transported': Y_test_pred
})

resultat_df.to_csv('resultat_nb.csv', index=False)

Meilleurs paramètres :  {'var_smoothing': 1.0}
